# Multi-Seed Training for Statistical Validation

**Goal:** Re-run the two most important models - **Baseline (0% adv)** and **Best Defended (25% adv)** - with 3 different random seeds to quantify result variance.

This is required for robust evaluation, which expects `mean +/- std` rather than single-seed results.

**Seeds:** 1, 2, 3  
**Models per seed:** 2 (Baseline + 25% Defended)  
**Total runs:** 6  

**Outputs saved to** `/kaggle/working/models/multi_seed/`:
- `baseline_seed_1.pt`, `baseline_seed_2.pt`, `baseline_seed_3.pt`, `defended_25pct_seed_1.pt`, `defended_25pct_seed_2.pt`, `defended_25pct_seed_3.pt`

### 1. Environment Setup

In [ ]:
import os, sys, subprocess, glob, shutil

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'transformers', 'sentencepiece', 'tqdm'], check=True)

WORKING_DIR = '/kaggle/working'

# Auto-detect dataset paths regardless of Kaggle username
def find_dataset(name, filename):
    matches = glob.glob(f'/kaggle/input/**/{name}/{filename}', recursive=True)
    return matches[0] if matches else None

def find_dataset_dir(name, subdir):
    matches = glob.glob(f'/kaggle/input/**/{name}/{subdir}', recursive=True)
    return matches[0] if matches else None

ADV_DATA_PATH  = find_dataset('sage-data', 'adv_training_pairs.json')
MODEL_PATH     = find_dataset('sage-model', 'baseline_epoch_3.pt')
SRC_PATH       = find_dataset_dir('sage-src', 'src')

# Copy src so Python can import it
if SRC_PATH and not os.path.exists(f'{WORKING_DIR}/src'):
    shutil.copytree(SRC_PATH, f'{WORKING_DIR}/src')
sys.path.insert(0, WORKING_DIR)

os.makedirs(f'{WORKING_DIR}/models/multi_seed', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/results', exist_ok=True)

print('Environment ready.')
print(f'Adv Data: {ADV_DATA_PATH}')
print(f'Baseline Model: {MODEL_PATH}')
print(f'Src: {SRC_PATH}')

### 2. Verify GPU

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU!')

### 3. Multi-Seed Training Loop

For each seed, we train two models:
- **Baseline**: 0% adversarial mix (only clean HH-RLHF data)
- **25% Defended**: 25% adversarial mix (our best ablation point)

Both are initialized from the same pre-trained `baseline_epoch_3.pt` checkpoint so the only variable is the seed and the adversarial ratio.

In [ ]:
import src.robustness.adversarial_training as adv_train

# Set paths for Kaggle environment
adv_train.ADV_DATA_PATH   = ADV_DATA_PATH
adv_train.BASE_CHECKPOINT = MODEL_PATH
adv_train.RESULTS_DIR     = f'{WORKING_DIR}/results'

SEEDS = [1, 2, 3]
RATIOS = [
    (0.00, 'baseline'),
    (0.25, 'defended_25pct'),
]

all_results = []

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'SEED {seed}')
    print(f'{"="*60}')
    
    for ratio, label in RATIOS:
        out_path = f'{WORKING_DIR}/models/multi_seed/{label}_seed_{seed}.pt'
        adv_train.OUTPUT_DIR = f'{WORKING_DIR}/models/multi_seed'
        
        print(f'\nTraining {label} (adv_ratio={ratio:.0%}, seed={seed})...')
        
        # Run with a single ratio and explicit seed
        result = adv_train.run_ablation_study(
            adv_ratios=[ratio],
            epochs=3,
            device=device,
            seed=seed
        )
        
        # Rename checkpoint to include seed info
        # run_ablation_study saves as reward_model_adv_{ratio*100:.0f}pct.pt
        default_name = f'{WORKING_DIR}/models/multi_seed/reward_model_adv_{ratio*100:.0f}pct.pt'
        if os.path.exists(default_name):
            os.rename(default_name, out_path)
            print(f'Saved: {out_path}')
        
        all_results.append({
            'seed': seed,
            'label': label,
            'adv_ratio': ratio,
            'checkpoint': out_path,
            'final_train_acc': result[0]['final_train_acc'] if result else None
        })

print('\nAll training runs complete!')

### 4. Summary & Save

In [ ]:
import json

print('\n--- Multi-Seed Training Summary ---')
print(f'{"Seed":<6} {"Model":<20} {"Ratio":<8} {"Final Acc"}')
print('-' * 55)
for r in all_results:
    acc_str = f"{r['final_train_acc']:.4f}" if r['final_train_acc'] else 'N/A'
    print(f"{r['seed']:<6} {r['label']:<20} {r['adv_ratio']:.0%}      {acc_str}")

# Save summary JSON for download
summary_path = f'{WORKING_DIR}/results/multi_seed_training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSummary saved to: {summary_path}')
print('\nDownload the /kaggle/working/ directory to get all 6 checkpoints!')
print('Expected files:')
for seed in [1, 2, 3]:
    for label in ['baseline', 'defended_25pct']:
        print(f'  models/multi_seed/{label}_seed_{seed}.pt')